# 06 - Agent Orchestrator End-to-End

Test the full agent workflow: `pptx_agent_start` -> `pptx_agent_respond` -> `pptx_agent_execute` -> `pptx_agent_rollback`.

**Requirements:** Set one of the following env-var combos before launching Jupyter:

| Provider | Variables |
|---|---|
| Anthropic (default) | `ANTHROPIC_API_KEY` |
| OpenAI | `PPTX_LLM_PROVIDER=openai` + `OPENAI_API_KEY` |
| Azure OpenAI | `PPTX_LLM_PROVIDER=azure_openai` + `AZURE_OPENAI_API_KEY` + `AZURE_OPENAI_ENDPOINT` + `AZURE_OPENAI_DEPLOYMENT` |

In [ ]:
import os
import sys
from pathlib import Path
from pprint import pprint

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "python").exists():
    if (REPO_ROOT.parent / "python").exists():
        REPO_ROOT = REPO_ROOT.parent
    else:
        raise RuntimeError("Run notebook from repo root or notebooks/ directory")

PYTHON_ROOT = REPO_ROOT / "python"
if str(PYTHON_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTHON_ROOT))

from service import PowerPointService
from errors import BridgeError

ARTIFACT_DIR = REPO_ROOT / "artifacts" / "notebook-tests"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Check LLM config
provider = os.getenv("PPTX_LLM_PROVIDER", "anthropic")
has_key = bool(
    os.getenv("PPTX_LLM_API_KEY")
    or os.getenv("ANTHROPIC_API_KEY")
    or os.getenv("OPENAI_API_KEY")
    or os.getenv("AZURE_OPENAI_API_KEY")
)
print(f"LLM provider: {provider}")
print(f"API key configured: {has_key}")
if not has_key:
    print("WARNING: No LLM API key found. Agent tools will fail.")
    print("Set ANTHROPIC_API_KEY, OPENAI_API_KEY, or AZURE_OPENAI_API_KEY.")

## Build a presentation to transform

Create a simple 3-slide deck simulating a company pitch.

In [ ]:
service = PowerPointService()
engine_info = service.dispatch("pptx_get_engine_info", {})
print("Engine:", engine_info.get("engine"))
print("Orchestrator active:", service.orchestrator is not None)

# Create deck
create = service.dispatch("pptx_create_presentation", {"width": "10in", "height": "5.625in"})
pid = create["presentation_id"]

# Discover layouts
layouts = service.dispatch("pptx_get_layouts", {"presentation_id": pid})["layouts"]
layout_names = [l["name"] for l in layouts]
print("Available layouts:", layout_names)

# Pick a layout for content slides
layout_name = layout_names[0]
print(f"Using layout: '{layout_name}'")

In [ ]:
# Add 3 slides
for i in range(3):
    service.dispatch("pptx_add_slide", {"presentation_id": pid, "layout_name": layout_name})

state = service.dispatch("pptx_get_presentation_state", {"presentation_id": pid})
print(f"Slide count: {state['slide_count']}")

# Populate slides with placeholder text
slide_content = {
    1: "Acme Corp - Company Overview",
    2: "Our Mission and Values",
    3: "Q3 2025 Financial Summary",
}
for slide_idx, text in slide_content.items():
    placeholders = service.dispatch(
        "pptx_get_placeholders", {"presentation_id": pid, "slide_index": slide_idx}
    )["placeholders"]
    # Try to set the first available placeholder
    for ph in placeholders:
        try:
            service.dispatch("pptx_set_placeholder_text", {
                "presentation_id": pid,
                "slide_index": slide_idx,
                "placeholder_name": ph["name"],
                "text_content": text,
            })
            print(f"  Slide {slide_idx}: set '{ph['name']}' to '{text}'")
            break
        except BridgeError:
            continue

# Add a text box on slide 2 for extra content
service.dispatch("pptx_add_text_box", {
    "presentation_id": pid,
    "slide_index": 2,
    "left": "1in",
    "top": "2in",
    "width": "8in",
    "height": "2in",
    "text_content": "We believe in innovation, quality, and customer success. Our team of 500+ engineers delivers world-class solutions.",
})
print("\nPresentation built.")

## Step 1: `pptx_agent_start`

Give the agent a natural language transformation query.

In [ ]:
start_result = service.dispatch("pptx_agent_start", {
    "presentation_id": pid,
    "query": "Add a professional blue and white color scheme to all slides. Set slide backgrounds to a light blue (#E8F0FE), make all title text dark navy (#1A237E) and bold, and add a thin rectangle accent bar at the bottom of each slide in navy.",
})

task_id = start_result["task_id"]
print(f"Task ID: {task_id}")
print(f"State: {start_result['state']}")
print(f"Estimated steps: {start_result.get('estimated_steps', 'N/A')}")
print(f"\nAnalysis:")
pprint(start_result.get("analysis", {}))

if start_result.get("questions"):
    print(f"\nClarifying questions ({len(start_result['questions'])}):")
    for q in start_result["questions"]:
        print(f"  [{q['question_id']}] {q['text']}")
        for c in q.get("choices", []):
            print(f"    {c}")
else:
    print("\nNo clarifying questions - plan generated directly.")

## Step 2: `pptx_agent_respond` (if needed)

If the agent returned questions, answer them here. Skip this cell if state is already `ready`.

In [ ]:
if start_result["state"] == "clarifying" and start_result.get("questions"):
    # Auto-answer: pick the first choice for each question
    answers = []
    for q in start_result["questions"]:
        first_choice = q["choices"][0] if q.get("choices") else "Yes"
        answer_text = first_choice.split(":")[0].strip()  # e.g. "A" from "A: Some option"
        answers.append({"question_id": q["question_id"], "answer": answer_text})
        print(f"  Answering '{q['question_id']}' with: {answer_text}")

    respond_result = service.dispatch("pptx_agent_respond", {
        "task_id": task_id,
        "answers": answers,
    })
    print(f"\nState after respond: {respond_result['state']}")
    if respond_result["state"] == "ready":
        print(f"Plan steps: {respond_result.get('step_count', 'N/A')}")
        for step in respond_result.get("steps", []):
            print(f"  {step['step_id']}: {step['description']}")
else:
    print("No questions to answer - plan is already generated.")

## Step 3: Check status before execution

In [ ]:
status = service.dispatch("pptx_agent_status", {"task_id": task_id})
print("Current status:")
pprint(status)

## Step 4: `pptx_agent_execute`

Execute the plan. A rollback snapshot is created automatically.

In [ ]:
exec_result = service.dispatch("pptx_agent_execute", {
    "task_id": task_id,
    "confirm": True,
})

print(f"State: {exec_result['state']}")
print(f"Steps executed: {exec_result.get('steps_executed', 0)}/{exec_result.get('steps_total', 0)}")
print(f"Rollback available: {exec_result.get('rollback_available', False)}")

if exec_result["state"] == "complete":
    print(f"\nSummary: {exec_result.get('summary', '')}")
    print("\nVerification:")
    pprint(exec_result.get("verification", {}).get("position_check", {}).get("summary", {}))
    pprint(exec_result.get("verification", {}).get("content_check", {}).get("summary", {}))
elif exec_result["state"] == "failed":
    print(f"\nError: {exec_result.get('error', 'unknown')}")

print("\nExecution log:")
for step in exec_result.get("execution_log", []):
    status_icon = {"done": "+", "failed": "X", "skipped": "-", "pending": "."}[step["status"]]
    err = f" ({step['error']})" if step.get("error") else ""
    print(f"  [{status_icon}] {step['step_id']}: {step['description']}{err}")

## Step 5: Save the result and inspect

In [ ]:
output_path = str((ARTIFACT_DIR / "nb06_agent_result.pptx").resolve())
service.dispatch("pptx_save_presentation", {
    "presentation_id": pid,
    "output_path": output_path,
})
print(f"Saved result to: {output_path}")

# Show final state
final_state = service.dispatch("pptx_get_presentation_state", {"presentation_id": pid})
print(f"\nFinal slide count: {final_state['slide_count']}")
for slide in final_state.get("slides", []):
    print(f"  Slide {slide['index']}: layout={slide.get('layout', 'N/A')}, title={slide.get('title', 'N/A')}")

## Step 6: `pptx_agent_rollback` (optional)

Demonstrate rollback: restore the presentation to its pre-execution state.

In [ ]:
rollback_result = service.dispatch("pptx_agent_rollback", {"task_id": task_id})
print(f"State after rollback: {rollback_result['state']}")
print(f"Message: {rollback_result['message']}")

# Verify the presentation is back to the original state
restored_state = service.dispatch("pptx_get_presentation_state", {"presentation_id": pid})
print(f"\nRestored slide count: {restored_state['slide_count']}")

# Save the rolled-back version for comparison
rollback_path = str((ARTIFACT_DIR / "nb06_agent_rolled_back.pptx").resolve())
service.dispatch("pptx_save_presentation", {
    "presentation_id": pid,
    "output_path": rollback_path,
})
print(f"Saved rolled-back version to: {rollback_path}")

## Step 7: Diff the before/after

Open the executed result and diff it against the rolled-back (original) version.

In [ ]:
# Open the saved result as a second presentation for diff
open_result = service.dispatch("pptx_open_presentation", {"file_path": output_path})
pid_result = open_result["presentation_id"]

diff = service.dispatch("pptx_diff_presentations", {
    "presentation_id_a": pid,         # rolled-back original
    "presentation_id_b": pid_result,   # agent-modified version
    "deep_diff": True,
})
print("Diff summary:", diff["summary"])
print("Slide counts:", diff["slide_count_diff"])
if diff["modified_slides"]:
    for ms in diff["modified_slides"]:
        print(f"\nSlide {ms['slide_index']} changes:")
        for ch in ms["changes"][:10]:
            pprint(ch)

service.dispatch("pptx_close_presentation", {"presentation_id": pid_result})

## Cleanup

In [ ]:
# Cancel the agent task to release snapshot
try:
    service.dispatch("pptx_agent_cancel", {"task_id": task_id})
    print("Agent task cancelled.")
except BridgeError as exc:
    print(f"Cancel skipped: {exc.message}")

service.dispatch("pptx_close_presentation", {"presentation_id": pid})
service.shutdown()
print("All sessions closed.")